# Chiến thuật phân loại review rác — ViSpamReviews v2 (v3)

**Mục tiêu:** tối đa **macro-F1** trên bảng xếp hạng.

**Dữ liệu:** `Final project/Data/` — ViSpamReviews v2 (bản có metadata), dùng trong bài báo [arXiv:2207.14636](https://arxiv.org/abs/2207.14636) và bản mở rộng [arXiv:2405.13292](https://arxiv.org/abs/2405.13292).

**Kết quả hiện tại:** Kaggle **0.78** (macro-F1), vượt PhoBERT của bài báo (72.17%).

> Tài liệu này chỉ trình bày **chiến thuật**; code nằm trong các notebook ở `src/`.

## Cập nhật quan trọng ở v3

1. **Metric là macro-F1**, không phải accuracy (xem mục 1.3).
2. **Class weight ở tầng 1** với mode `sqrt` giúp tăng điểm.
3. **Stacking tự so sánh nhiều bộ OOF** (`default`, `balanced`, `sqrt`).
4. Kết quả Kaggle: 0.72 → 0.73 → 0.74 → 0.77 → **0.78**.

## 1. Bài toán, dữ liệu & metric

### 1.1 Nhãn

| `spam_label` | Ý nghĩa |
|---|---|
| `0` | No-spam |
| `1` | Fake review |
| `2` | Review on brand only |
| `3` | Non-review |

### 1.2 Phân bố

| Split | Số dòng | class 0 | class 1 | class 2 | class 3 |
|---|---:|---:|---:|---:|---:|
| train | 14,306 | 10,512 (73.48%) | 211 (1.47%) | 1,103 (7.71%) | 2,480 (17.34%) |
| dev | 1,590 | 1,183 (74.40%) | 17 (1.07%) | 120 (7.55%) | 270 (16.98%) |
| test | 3,974 | không nhãn | — | — | — |

### 1.3 Vì sao metric là macro-F1, không phải accuracy

Leaderboard không ghi metric, nhưng các điểm số chỉ khớp với macro-F1:

| Submission | Kaggle | Dev accuracy | Dev macro-F1 |
|---|---:|---:|---:|
| LightGBM đơn | 0.72 | 0.8937 | 0.6719 |
| Stacking 5 model (tối ưu accuracy) | 0.73 | 0.9044 | 0.7258 |
| Stacking 3 model (tối ưu accuracy) | 0.74 | 0.9069 | 0.7364 |

Nếu là accuracy, điểm phải ~0.89–0.91. Thứ tự tăng/giảm cũng khớp macro-F1, không khớp accuracy.

**Kết luận:** mọi bước từ đây tối ưu **macro-F1**; accuracy chỉ để tham khảo.

**Hệ quả:** class 1 (1.5%) và class 2 (7.7%) có trọng số ngang class 0. Phải ưu tiên recall hai lớp hiếm, chấp nhận giảm accuracy.

## 2. Kiến trúc & các notebook

```
TẦNG 0 — DỮ LIỆU & ĐẶC TRƯNG
  src/preprocessing.ipynb
      → X_*.npz, dense_*.npy, clean_*.parquet, y_*.npy, preprocessors.joblib

TẦNG 1A — BASE MODEL + OOF
  src/train_baseline.ipynb      → model full-train + probs dev/test
  src/oof_probabilities.ipynb   → OOF 5-fold, không class weight
  src/oof_class_weight.ipynb    → OOF 5-fold, balanced + sqrt

TẦNG 2 — STACKING
  src/stacking.ipynb
      → tự chọn variant, tune C + prior, xuất submission

PHỤ
  predict_lightgbm.ipynb        → submission từ model đơn (tham khảo)
```

**Nguyên tắc:** `preprocessing.ipynb` là "nguồn chân lý" về đặc trưng. Các notebook sau chỉ đọc artifacts, không tự tạo đặc trưng khác.

**Thêm PhoBERT sau này:** tạo nhánh mới, **không sửa** `X_*.npz`; chỉ thêm `comment_seg` và xác suất PhoBERT vào stacking.

## 3. Tiền xử lý

### 3.1 Ba dạng text

| Cột | Dùng cho | Đặc điểm |
|---|---|---|
| `comment_norm` | **PhoBERT** | NFC, unescape HTML, giữ dấu/hoa/emoji |
| `comment_model` | TF-IDF | lower-case, thay URL/email/SĐT bằng token, gộp ký tự lặp ≥3 |
| `comment_seg` (chưa có) | PhoBERT | tách từ bằng underthesea/VnCoreNLP |

### 3.2 Quy tắc

1. Đọc CSV `utf-8-sig`; comment chứa `\r\n` bên trong ngoặc kép.
2. `html.unescape` + `unicodedata.normalize("NFC")`.
3. Parse `categories` bằng `ast.literal_eval` (không phải JSON).
4. **Không bỏ stopword** trước PhoBERT (phá phủ định, mất ngữ cảnh).
5. Không dùng đặc trưng suy ra từ nhãn.

## 4. Đặc trưng dùng chung

**52 đặc trưng dense:**
- Bề mặt: độ dài, số từ, số dòng, ký tự lặp, emoji, chữ hoa, chữ số, tỷ lệ dấu.
- Spam: URL/SĐT/email, run ký tự lặp, từ viết hoa.
- Lexicon: dịch vụ / sản phẩm / quảng cáo.
- Overlap: Jaccard/coverage giữa comment và `product_name`/`description`.
- Metadata: `rating`, `log_num_sold`, `log_num_review`, one-hot `category`, multi-label `categories`.
- Corpus (không nhãn): `comment_dup_count`, `comment_dup_products`, `product_freq`, `product_mean_rating`.

**Giá trị của metadata (đo được):** LinearSVC trên TF-IDF + metadata đạt **0.8937** dev accuracy; bỏ metadata còn **0.8704** → metadata đóng góp **+2.33 điểm**.

Với PhoBERT, 52 đặc trưng này được fusion ở head phân loại.

## 5. Biểu diễn văn bản

### 5.1 TF-IDF (nhánh hiện tại)

| Khối | Cột | Cấu hình |
|---|---|---|
| comment_word | `comment_model` | word 1–2 gram, min_df=3, 200k |
| comment_char | `comment_model` | char_wb 2–5 gram, min_df=3, 200k |
| product_name | `product_name_model` | word 1–2 gram, min_df=2, 50k |
| description | `description_model` | word 1 gram, min_df=5, 80k |

Kết quả: **98,713 đặc trưng**, train ~7.5M nnz.

### 5.2 PhoBERT (nhánh sẽ thêm)

- `vinai/phobert-base`, `max_len=256` (hoặc 128 nếu thiếu VRAM).
- Input `comment_seg` (đã tách từ), không lower-case, giữ dấu.
- Fusion `[CLS]` (768) ‖ `Linear(52→64)` → Linear(4).

## 6. Tầng 1A — base model & OOF

### 6.1 Năm base model

| Model | Dev accuracy | Dev macro-F1 |
|---|---:|---:|
| LightGBM | 0.8937 | 0.6719 |
| Logistic Regression | 0.8899 | 0.6950 |
| LinearSVC (calibrated) | 0.8881 | 0.6822 |
| SGDClassifier | 0.8811 | 0.6916 |
| ComplementNB | 0.8302 | 0.4660 |

### 6.2 OOF 5-fold (macro-F1)

| Model | default (no weight) | balanced | **sqrt** |
|---|---:|---:|---:|
| LightGBM | 0.7114 | **0.7335** | 0.7222 |
| Logistic Regression | 0.7211 | 0.7240 | **0.7433** |
| LinearSVC | 0.7113 | 0.7098 | 0.7132 |
| SGD | 0.6047 | **0.6219** | 0.6177 |
| ComplementNB | 0.5095 | 0.4867 | **0.6051** |

**Kết luận:**
- `sqrt` tốt hơn `balanced` cho hầu hết model.
- `balanced` phá ComplementNB (accuracy 0.63).
- LogReg + `sqrt` là model đơn tốt nhất (0.7433).

### 6.3 Cơ chế weight

```
w_c = n / (k · count_c)          # balanced
w_c = sqrt(w_c^balanced)          # sqrt
```

Với dữ liệu này: class 1 ≈ 17× (balanced) hoặc ≈ 4.1× (sqrt). Class 1 chỉ có 211 mẫu nên weight quá lớn dễ over-predict.

## 7. Tầng 2 — stacking

### 7.1 Meta features

```
[ log(p) của 5 model ]  → 20 chiều
[ rating, log_num_sold, log_num_review ] → 3 chiều
= 23 chiều
```

Dùng `log(p)` thay vì `p` để ổn định khi xác suất sát 0/1.

### 7.2 Meta-learner

- **Logistic Regression multinomial**, `class_weight=None`.
- Grid `C ∈ {0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 10.0}`.
- Thử `class_weight='balanced'` → **làm giảm** macro-F1 (0.72 vs 0.756); prior tuning đã thay thế tốt hơn.
- **Prior tuning** `w=[w0..w3]` bằng coordinate ascent theo **macro-F1** trên dev.

### 7.3 Tự so sánh variant

| Variant | OOF | dev/test |
|---|---|---|
| default | `probs/oof/` | `probs/no-oof/` |
| balanced | `probs/oof_balanced/` | cùng thư mục |
| sqrt | `probs/oof_sqrt/` | cùng thư mục |

Kết quả tốt nhất:

| Variant | `C` | Prior | Dev macro-F1 | Dev accuracy |
|---|---:|---|---:|---:|
| **sqrt** | **0.1** | **[0.5, 0.8, 1.1, 0.9]** | **0.7671** | 0.8956 |
| default | 0.01 | [0.5, 0.9, 1.25, 1.0] | 0.7564 | 0.8912 |
| balanced | 0.03 | [0.6, 1.0, 1.25, 1.0] | 0.7549 | 0.8925 |

Prior tuning đóng góp lớn: với `sqrt`, từ **0.7321** (không prior) → **0.7671**.

## 8. Kết quả thực tế trên leaderboard

| Bước | Dev macro-F1 | Kaggle |
|---|---:|---:|
| LightGBM đơn | 0.6719 | 0.72 |
| Stacking 5 model (accuracy) | 0.7258 | 0.73 |
| Stacking 3 model (accuracy) | 0.7364 | 0.74 |
| Stacking 5 model (macro-F1) | 0.7564 | 0.77 |
| **+ class weight `sqrt` tầng 1** | **0.7671** | **0.78** |

**So với bài báo:** PhoBERT text-only đạt macro-F1 **72.17%**; pipeline hiện tại **76.71% dev / 0.78 Kaggle** → vượt khoảng **5–6 điểm**.

Phân bố nhãn dự đoán test (submission 0.78):

| Nhãn | Tỷ lệ |
|---|---:|
| 0 | 72.2% |
| 1 | 1.1% |
| 2 | 9.0% |
| 3 | 17.8% |

Rất sát phân bố train (73.5 / 1.5 / 7.7 / 17.3).

## 9. Ablation & phân tích lỗi

### 9.1 Ablation với variant `sqrt`

| Bỏ model | Dev macro-F1 | Chênh lệch |
|---|---:|---:|
| LightGBM | 0.7388 | −0.028 |
| ComplementNB | 0.7478 | −0.019 |
| Logistic | 0.7658 | −0.001 |
| SGD | 0.7647 | −0.002 |
| LinearSVC | 0.7641 | −0.003 |
| Đầy đủ | **0.7671** | — |

→ LightGBM và ComplementNB đóng góp lớn; 3 model còn lại gần như trung tính.

### 9.2 Confusion matrix trên dev

| | pred_0 | pred_1 | pred_2 | pred_3 |
|---|---:|---:|---:|---:|
| true_0 | 1098 | 1 | 59 | 25 |
| true_1 | 6 | 9 | 0 | 2 |
| true_2 | 44 | 0 | **74** | 2 |
| true_3 | 21 | 0 | 6 | 243 |

- Class 2: recall **0.617**, precision 0.532 → điểm yếu chính.
- Class 1: recall 0.529, precision 0.900.
- Class 0/3 ổn định (~0.93 / 0.90).

**Đây là chỗ PhoBERT có khả năng cải thiện rõ nhất**, vì class 2 cần hiểu ngữ nghĩa "chỉ nói về thương hiệu/dịch vụ".

## 10. Nhánh PhoBERT

### 10.1 Dùng lại gì

| Thành phần | Trạng thái |
|---|---|
| Raw data, nhãn, split, ID | Dùng nguyên |
| `comment_norm` | Đã đúng cho PhoBERT |
| 52 dense features | Fusion ở head |
| Fold split (seed 42) | Dùng chung |
| Stacking, prior, submission | Dùng nguyên, thêm 1 base model |
| `X_*.npz`, TF-IDF, 5 model cũ | **Không đụng** |

### 10.2 Việc mới

1. Tách từ: `comment_norm` → `comment_seg`.
2. Tokenizer PhoBERT, `max_len=256`.
3. Cài torch CUDA + transformers.
4. Fine-tune có class weight cho macro-F1.
5. OOF 3 fold, lưu `{oof,dev,test}` đúng định dạng.
6. Thêm `phobert` vào stacking.

### 10.3 Cấu hình cho RTX 2050 4 GB

| Tham số | Giá trị |
|---|---|
| Model | `vinai/phobert-base` |
| `max_len` | 128 (chính) hoặc 256 nếu đủ VRAM |
| Batch | 8 (max_len 128) |
| Optimizer | AdamW, lr 2e-5, weight decay 0.01 |
| Epochs | 3–5, early stopping theo macro-F1 trên dev |
| Precision | AMP fp16/bf16 |
| Chống OOM | gradient checkpointing, `empty_cache()` |
| Mất cân bằng | class weight nhẹ; ưu tiên macro-F1 |

### 10.4 Chi phí

| Việc | Thời gian |
|---|---|
| Cài môi trường | ~20 phút |
| Tách từ | ~5 phút |
| Fine-tune 1 fold | ~45–60 phút |
| OOF 3 fold | ~3 giờ |

Kỳ vọng: +1–2 điểm macro-F1, chủ yếu ở class 2.

## 11. Môi trường GPU

| Thành phần | Giá trị |
|---|---|
| GPU | NVIDIA RTX 2050 (laptop), 4 GB VRAM |
| Kiến trúc | Ampere, sm_86 → fp16 + bf16 |
| Driver | 616.56, CUDA UMD 13.4 |
| Python | 3.12 |
| Có sẵn | pandas, numpy, scikit-learn, scipy, lightgbm 4.7.0 |
| Thiếu/hỏng | torch (chỉ còn dist-info, thiếu package), transformers, sentencepiece, datasets, accelerate |

Cài lại:

```powershell
python -m pip uninstall -y torch torchvision torchaudio
python -m pip install torch --index-url https://download.pytorch.org/whl/cu126
python -m pip install transformers sentencepiece datasets accelerate
```

Kiểm tra:

```python
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.is_available())
```

Lưu ý: TensorFlow 2.21 trên Windows native **không hỗ trợ GPU** → dùng PyTorch.

## 12. Artifacts & quy trình chạy

### 12.1 Cấu trúc

```
artifacts/
├── sparse/      X_{train,dev,test}.npz
├── dense/       dense_*.npy
├── clean/       clean_*.parquet
├── labels/      y_*.npy
├── ids/         test_ids.csv
├── meta/        preprocess_config.json
├── models/
│   ├── preprocessors.joblib
│   ├── classifiers/     base model full-train
│   └── meta/            meta_learner.joblib
└── probs/
    ├── no-oof/          dev/test từ model full-train
    ├── oof/             OOF + test fold-avg (không weight)
    ├── oof_balanced/    OOF + dev/test (balanced)
    └── oof_sqrt/        OOF + dev/test (sqrt)

submissions/
└── submission_stacking.csv
```

### 12.2 Thứ tự chạy

1. `src/preprocessing.ipynb`
2. `src/train_baseline.ipynb`
3. `src/oof_probabilities.ipynb`
4. `src/oof_class_weight.ipynb` (balanced, sqrt)
5. `src/stacking.ipynb`

### 12.3 Quy tắc

- OOF notebook có `SKIP_IF_EXISTS` — chạy lại an toàn.
- Stacking tự phát hiện variant; variant thiếu file bị bỏ qua.
- Chỉ cần ≥2 model mỗi variant để stacking chạy.

## 13. Ghi chú kỹ thuật (gotchas)

- **scipy sparse**: không dùng `len(X)`; dùng `X.shape[0]` (lỗi đã gặp).
- **Layout `probs/`**: dev/test của variant default nằm trong `probs/no-oof/`, OOF trong `probs/oof/`.
- **Không fit trên dev**: dev chỉ để chọn `C`, variant, prior.
- **Prior tuning trên dev** có thể lạc quan nhẹ; ưu tiên so sánh tương đối.
- **`class_weight='balanced'` ở meta phản tác dụng** — đã thử.
- **Class 1 chỉ 211 mẫu** — weight quá lớn làm precision sụp.
- **Metric là macro-F1** — đừng tối ưu accuracy.
- **Không đổi `X_*.npz`** khi thêm PhoBERT; chỉ thêm cột text mới.
- **Seed cố định 42** cho split và model.

## 14. Hướng tiếp theo

| Hướng | Chi phí | Kỳ vọng |
|---|---|---|
| **Mixed-mode stacking** — mỗi model lấy mode tốt nhất | ~2 phút | +0.5–1 điểm |
| **PhoBERT OOF** — nhánh mới | ~4 giờ | +1–2 điểm, mạnh ở class 2 |
| 10-fold OOF | ~2.5 giờ | Ổn định hơn, +0.2–0.5 |
| Seed averaging | ~1 giờ | +0.1–0.3 |

### Mixed-mode đề xuất

```
logreg         → sqrt      (0.7433)
lightgbm       → balanced  (0.7335)
linsvc         → sqrt      (0.7132)
complement_nb  → sqrt      (0.6051)
sgd            → balanced  (0.6219)
```

Vì mỗi model mạnh ở một mode khác nhau, trộn mode có thể tốt hơn dùng chung một variant.

## 15. Tài liệu tham khảo

- Dinh, Luu, Nguyen (2022). *Detecting Spam Reviews on Vietnamese E-commerce Websites*. ACIIDS 2022. [arXiv:2207.14636](https://arxiv.org/abs/2207.14636)
- Dinh & Luu (2024). *Metadata Integration for Spam Reviews Detection on Vietnamese E-commerce Websites*. [arXiv:2405.13292](https://arxiv.org/abs/2405.13292)
- Mã nguồn & dữ liệu gốc: [github.com/sonlam1102/vispamdetection](https://github.com/sonlam1102/vispamdetection) · HuggingFace [visolex/ViSpamReviews](https://huggingface.co/datasets/visolex/ViSpamReviews)
- PhoBERT: [github.com/VinAIResearch/PhoBERT](https://github.com/VinAIResearch/PhoBERT) · model `vinai/phobert-base`